# Weather Data Analysis — Banten

Notebook eksplorasi data cuaca historis untuk wilayah Banten. Analisis ini melengkapi dashboard Weather AI dengan alur kerja Data Analytics yang dapat diulang: pemeriksaan kualitas data, data cleaning, exploratory data analysis (EDA), analisis tren waktu, perbandingan kota, korelasi, dan ekspor ringkasan.

## Tujuan analisis
- Memahami struktur, rentang waktu, cakupan kota, dan kualitas dataset.
- Mengamati distribusi suhu, curah hujan, dan kecepatan angin.
- Membandingkan pola cuaca antar kota dan antar bulan.
- Menghasilkan insight deskriptif yang dapat diperiksa ulang.

> **Batasan:** notebook ini melakukan analisis deskriptif, bukan membuktikan hubungan sebab-akibat dan bukan evaluasi akurasi CNN-LSTM. Kategori curah hujan yang digunakan adalah aturan eksplorasi sederhana, bukan klaim standar resmi. Periksa metadata dan sumber dataset sebelum menggunakan hasil untuk keputusan operasional.

In [ ]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 1. Memuat dataset

Notebook mencari file `data/dataset_banten_clean.csv` dari direktori repository maupun saat notebook dijalankan dari folder `notebooks`.

In [ ]:
dataset_candidates = [
    Path.cwd() / "data" / "dataset_banten_clean.csv",
    Path.cwd().parent / "data" / "dataset_banten_clean.csv",
    Path.cwd().parent.parent / "data" / "dataset_banten_clean.csv",
]
DATA_PATH = next((path.resolve() for path in dataset_candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Jalankan notebook dari root repository "
        "atau folder notebooks, dan pastikan data/dataset_banten_clean.csv tersedia."
    )

df_raw = pd.read_csv(DATA_PATH, sep=None, engine="python")
print(f"Dataset: {DATA_PATH}")
print(f"Ukuran awal: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")
display(df_raw.head())

## 2. Data cleaning dan persiapan

Nama kolom dinormalisasi agar konsisten. Kolom tanggal dikonversi ke format datetime, kolom numerik dikonversi ke angka, dan baris tanpa tanggal atau nama kota yang valid dikeluarkan. Nilai ekstrem tidak langsung dihapus karena perlu pemeriksaan konteks terlebih dahulu.

In [ ]:
df = df_raw.copy()
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\\s+", "_", regex=True)
)

required_columns = {"date", "city"}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(
        f"Kolom wajib tidak ditemukan: {sorted(missing_required)}. "
        f"Kolom tersedia: {df.columns.tolist()}"
    )

rows_before_cleaning = len(df)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

numeric_candidates = [
    "temperature_2m_mean",
    "precipitation_sum",
    "windspeed_10m_max",
    "cape_proxy_radiation",
    "lat",
    "lon",
]
for column in numeric_candidates:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

df["city"] = df["city"].astype("string").str.strip()
df = df.dropna(subset=["date", "city"])
df = df[df["city"].ne("")]
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates()
df = df.sort_values(["city", "date"]).reset_index(drop=True)

print(f"Baris sebelum cleaning : {rows_before_cleaning:,}")
print(f"Baris setelah cleaning : {len(df):,}")
print(f"Baris duplikat dihapus: {duplicates_removed:,}")
print(f"Baris dikeluarkan karena tanggal/kota tidak valid: {rows_before_cleaning - len(df) - duplicates_removed:,}")
display(df.head())

## 3. Pemeriksaan kualitas data

Laporan berikut memperlihatkan tipe data, nilai kosong, persentase nilai kosong, dan jumlah nilai unik setiap kolom. Nilai kosong tidak otomatis diisi agar asumsi imputasi tidak mengubah karakter data tanpa alasan.

In [ ]:
quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
}).sort_values("missing_percent", ascending=False)

date_start = df["date"].min()
date_end = df["date"].max()
print(f"Rentang data: {date_start:%d %B %Y} — {date_end:%d %B %Y}")
print(f"Jumlah kota: {df['city'].nunique():,}")
print(f"Jumlah baris analisis: {len(df):,}")
display(quality_report)

In [ ]:
missing_to_plot = quality_report[quality_report["missing_values"] > 0]
if not missing_to_plot.empty:
    ax = missing_to_plot["missing_percent"].sort_values().plot(
        kind="barh", figsize=(9, 4), title="Persentase Nilai Kosong per Kolom"
    )
    ax.set_xlabel("Nilai kosong (%)")
    ax.set_ylabel("Kolom")
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ditemukan nilai kosong pada data hasil cleaning.")

## 4. Statistik deskriptif

Statistik berikut membantu melihat pusat data, variasi, rentang nilai, dan kemungkinan nilai yang perlu ditinjau. Interpretasi tetap mempertimbangkan satuan masing-masing variabel.

In [ ]:
climate_columns = [
    column for column in [
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_max",
        "cape_proxy_radiation",
    ]
    if column in df.columns
]

if climate_columns:
    display(df[climate_columns].describe().T)
else:
    print("Kolom cuaca numerik yang diharapkan tidak ditemukan.")

In [ ]:
plot_columns = [
    column for column in [
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_max",
    ]
    if column in df.columns and df[column].notna().any()
]

if plot_columns:
    fig, axes = plt.subplots(1, len(plot_columns), figsize=(5 * len(plot_columns), 4))
    if len(plot_columns) == 1:
        axes = [axes]
    for ax, column in zip(axes, plot_columns):
        values = df[column].dropna()
        if column == "precipitation_sum":
            values = np.log1p(values.clip(lower=0))
            label = "log(1 + curah hujan)"
        else:
            label = column.replace("_", " ").title()
        ax.hist(values, bins=30, edgecolor="black", alpha=0.8)
        ax.set_title(f"Distribusi {column.replace('_', ' ').title()}")
        ax.set_xlabel(label)
        ax.set_ylabel("Frekuensi")
    plt.tight_layout()
    plt.show()

## 5. Pola cuaca berdasarkan bulan

Data dikelompokkan berdasarkan bulan kalender. Nilai yang ditampilkan adalah rata-rata per baris kota-hari dalam dataset, bukan total curah hujan seluruh Provinsi Banten. Dengan begitu, bulan dengan lebih banyak kota atau hari tidak otomatis dibaca sebagai total hujan wilayah.

In [ ]:
monthly_metrics = [
    column for column in [
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_max",
    ]
    if column in df.columns
]

df_month = df.copy()
df_month["month_number"] = df_month["date"].dt.month
monthly_profile = (
    df_month.groupby("month_number")[monthly_metrics]
    .mean()
    .reindex(range(1, 13))
)
monthly_profile.index = [calendar.month_abbr[index] for index in monthly_profile.index]
display(monthly_profile)

if monthly_metrics:
    fig, axes = plt.subplots(1, len(monthly_metrics), figsize=(5 * len(monthly_metrics), 4))
    if len(monthly_metrics) == 1:
        axes = [axes]
    for ax, column in zip(axes, monthly_metrics):
        ax.plot(monthly_profile.index, monthly_profile[column], marker="o")
        ax.set_title(f"Rata-rata {column.replace('_', ' ').title()}")
        ax.set_xlabel("Bulan")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 6. Perbandingan antar kota

Ringkasan kota menggunakan rata-rata suhu, rata-rata curah hujan per baris kota-hari, median kecepatan angin maksimum, dan persentase hari dengan curah hujan di atas 0,1 mm. Persentase hari hujan dan kategori intensitas merupakan definisi eksplorasi yang dipakai dalam notebook ini.

In [ ]:
aggregation = {}
if "temperature_2m_mean" in df.columns:
    aggregation["temperature_2m_mean"] = "mean"
if "precipitation_sum" in df.columns:
    aggregation["precipitation_sum"] = "mean"
if "windspeed_10m_max" in df.columns:
    aggregation["windspeed_10m_max"] = "median"

city_agg = {"date": "count", **aggregation}
city_summary = (
    df.groupby("city").agg(city_agg)
    .rename(columns={"date": "records"})
)

if "precipitation_sum" in df.columns:
    wet_day_rate = (
        df.assign(is_wet_day=df["precipitation_sum"] > 0.1)
        .groupby("city")["is_wet_day"]
        .mean()
        .mul(100)
        .rename("wet_day_percent")
    )
    city_summary = city_summary.join(wet_day_rate)

display(city_summary.sort_values("records", ascending=False).head(15))

if "precipitation_sum" in city_summary.columns:
    top_rain = city_summary.sort_values("precipitation_sum", ascending=False).head(10)
    ax = top_rain["precipitation_sum"].sort_values().plot(
        kind="barh", figsize=(9, 5), title="10 Kota dengan Rata-rata Curah Hujan Tertinggi"
    )
    ax.set_xlabel("Rata-rata curah hujan per baris kota-hari (mm)")
    ax.set_ylabel("Kota")
    plt.tight_layout()
    plt.show()

## 7. Analisis korelasi

Korelasi Pearson digunakan sebagai gambaran hubungan linear antarvariabel numerik. Korelasi tidak membuktikan sebab-akibat. Kolom `cape_proxy_radiation` diperlakukan sebagai fitur proxy sebagaimana penamaannya dalam kode aplikasi, bukan otomatis dianggap sebagai pengukuran radiasi atau CAPE yang tervalidasi.

In [ ]:
correlation_columns = [
    column for column in [
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_max",
        "cape_proxy_radiation",
    ]
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column])
]
correlation_columns = [
    column for column in correlation_columns
    if df[column].notna().sum() > 1 and df[column].nunique(dropna=True) > 1
]

if len(correlation_columns) >= 2:
    correlation_matrix = df[correlation_columns].corr(method="pearson")
    display(correlation_matrix)

    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(correlation_matrix, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(correlation_columns)))
    ax.set_yticks(range(len(correlation_columns)))
    ax.set_xticklabels(correlation_columns, rotation=45, ha="right")
    ax.set_yticklabels(correlation_columns)
    for row in range(len(correlation_columns)):
        for column in range(len(correlation_columns)):
            ax.text(column, row, f"{correlation_matrix.iloc[row, column]:.2f}",
                    ha="center", va="center", fontsize=9)
    ax.set_title("Matriks Korelasi Pearson")
    fig.colorbar(image, ax=ax, label="Koefisien korelasi")
    plt.tight_layout()
    plt.show()
else:
    print("Kolom numerik bervariasi tidak cukup untuk menghitung korelasi.")

## 8. Pemeriksaan nilai ekstrem dengan metode IQR

Metode interquartile range (IQR) digunakan untuk menandai nilai yang jauh dari sebaran tengah. Baris yang terdeteksi **tidak dihapus**. Nilai tersebut dapat merupakan kejadian cuaca ekstrem yang valid atau kesalahan pencatatan, sehingga perlu divalidasi dengan sumber data.

In [ ]:
outlier_rows = []
for column in climate_columns:
    values = df[column].dropna()
    if len(values) < 4:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())
    outlier_rows.append({
        "column": column,
        "lower_iqr_bound": lower_bound,
        "upper_iqr_bound": upper_bound,
        "flagged_values": outlier_count,
        "flagged_percent": outlier_count / len(values) * 100,
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

## 9. Ringkasan insight otomatis

Insight di bawah dihitung dari dataset yang berhasil dibaca. Hasilnya bersifat deskriptif dan dapat berubah ketika dataset diperbarui. Periksa kembali cakupan tanggal dan jumlah catatan tiap kota sebelum membandingkan hasil.

In [ ]:
print("RINGKASAN DATA CUACA BANTEN")
print("=" * 42)
print(f"Rentang tanggal : {date_start:%Y-%m-%d} sampai {date_end:%Y-%m-%d}")
print(f"Jumlah kota     : {df['city'].nunique()}")
print(f"Jumlah catatan  : {len(df):,}")

if "temperature_2m_mean" in df.columns:
    print(f"Rata-rata suhu  : {df['temperature_2m_mean'].mean():.2f} °C")

if "precipitation_sum" in df.columns:
    print(f"Rata-rata hujan : {df['precipitation_sum'].mean():.2f} mm per catatan kota-hari")
    city_rain_mean = df.groupby("city")["precipitation_sum"].mean().dropna()
    if not city_rain_mean.empty:
        print(f"Rata-rata hujan tertinggi: {city_rain_mean.idxmax()} ({city_rain_mean.max():.2f} mm)")

if "temperature_2m_mean" in df.columns:
    city_temp_mean = df.groupby("city")["temperature_2m_mean"].mean().dropna()
    if not city_temp_mean.empty:
        print(f"Rata-rata suhu tertinggi : {city_temp_mean.idxmax()} ({city_temp_mean.max():.2f} °C)")

if "precipitation_sum" in monthly_profile.columns:
    wettest_month = monthly_profile["precipitation_sum"].idxmax()
    print(f"Bulan dengan rata-rata hujan tertinggi: {wettest_month}")

print("\nCatatan: insight ini bukan prediksi dan bukan bukti hubungan sebab-akibat.")

## 10. Ekspor hasil analisis

Notebook menyimpan ringkasan kualitas data dan ringkasan per kota ke folder `analysis_outputs/` pada root repository. Folder hasil tidak perlu dimasukkan ke GitHub karena seluruh hasil dapat dibuat ulang dengan menjalankan notebook.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent.parent / "analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=True)
city_summary.to_csv(OUTPUT_DIR / "city_weather_summary.csv", index=True)
monthly_profile.to_csv(OUTPUT_DIR / "monthly_weather_profile.csv", index=True)
if not outlier_report.empty:
    outlier_report.to_csv(OUTPUT_DIR / "outlier_iqr_report.csv", index=False)

print(f"Hasil analisis tersimpan di: {OUTPUT_DIR}")
print("File: data_quality_report.csv, city_weather_summary.csv, monthly_weather_profile.csv")
if not outlier_report.empty:
    print("Tambahan: outlier_iqr_report.csv")

## Kesimpulan dan pekerjaan lanjutan

Gunakan tabel dan grafik di atas untuk menulis kesimpulan berdasarkan nilai aktual yang muncul saat notebook dijalankan. Pengembangan berikutnya yang disarankan:
1. Tambahkan data dictionary berisi definisi, satuan, sumber, dan frekuensi setiap variabel.
2. Validasi nilai ekstrem dengan sumber data cuaca yang tepercaya.
3. Buat evaluasi model terpisah menggunakan data uji yang tidak dipakai saat training, lalu laporkan MAE dan RMSE untuk suhu serta metrik yang sesuai untuk curah hujan.
4. Hindari kebocoran data waktu dengan memisahkan data training dan testing berdasarkan kronologi, bukan pengacakan baris.